# Decision Model Demo — Entity Resolution & Decision-Model-as-Judge

Live demo for the Decision Models deep-dive session (Edition #1).

Two scenarios, one decision model:
1. **Entity resolution** — same entity, yes/no + calibrated probability.
2. **Decision-model-as-judge** — same model, swapped question schema, judging a RAG answer against a rubric.

Then a **contrast**: the same entity-resolution question answered by prompting a generic LLM for JSON, timed side by side.

Both halves are **pluggable providers** (see the Config cell):

| | Open-source default (today) | Hosted/alternative (drop-in later) |
|---|---|---|
| Decision model | **Laya** (ModernBERT, 421M), local, runs on CPU/MPS/CUDA | TypeSafe Jev (`jev-latest`) via API, or Torchcast Decision |
| LLM contrast | Ollama, local (`qwen3.6`) | OpenAI or Claude via API |

The default path needs no API keys and no internet at demo time: everything runs against models already on this machine.

**Why Laya and not Torchcast Decision, on this hardware:** Torchcast Decision 4B's own docs are explicit that it requires its CUDA-only serving stack (`flash-linear-attention`, `causal-conv1d`) for *correct* results — it mixes causal and non-causal attention layers, and serving it via MLX, llama.cpp, or plain `transformers` on Apple Silicon applies the wrong attention mask and silently returns different (wrong) probabilities. There is no legitimate CPU/MPS path for it. Laya is a standard ModernBERT encoder with explicit, documented MPS support, so it's the one that's actually live here — not a simulated stand-in.

## Setup — run this **before** the session, not live

**Decision model (Laya, open-source default):**
```bash
python3.12 -m venv .laya-venv && source .laya-venv/bin/activate
pip install "laya[serve]"

LAYA_DEVICE=mps LAYA_HOST=127.0.0.1 LAYA_PORT=8012 laya-serve
# device: "mps" on Apple Silicon, "cuda" on an NVIDIA GPU, "cpu" otherwise -- all are correct, just different speeds
```
First start preloads three checkpoints (english/multilingual/typed-decisions) from Hugging Face and took several minutes here; subsequent starts are much faster once weights are cached locally.

**Decision model (Torchcast Decision 4B, documented but NOT usable on Apple Silicon):**
```bash
# Requires an NVIDIA GPU. Do not attempt on Apple Silicon -- see note in the intro cell above.
python3.12 -m venv .venv && source .venv/bin/activate
pip install "torch==2.9.1" --index-url https://download.pytorch.org/whl/cu128
pip install "transformers==5.8.1" "flash-linear-attention==0.5.2" accelerate safetensors "huggingface_hub[cli]" ninja packaging wheel setuptools
pip install --no-build-isolation "causal-conv1d>=1.7"

hf download torchcast-ai/torchcast-decision-4b --revision v1.3.1 --local-dir torchcast-decision-4b-v1.3
cd torchcast-decision-4b-v1.3 && sha256sum -c SHA256SUMS

python -m torchcast_decision.check .
python -m torchcast_decision.server --model . --host 127.0.0.1 --port 8011
```

**LLM contrast (Ollama, open-source default):**
```bash
ollama pull qwen3.6   # skip if already pulled -- check with `ollama list`
ollama serve          # skip if already running
```

**Future drop-in alternatives** (not needed for the default run — set these only to switch providers in the Config cell):
```bash
export TYPESAFE_API_KEY=...   # to switch DECISION_PROVIDER to "jev"
export OPENAI_API_KEY=...     # to switch LLM_PROVIDER to "openai"
export ANTHROPIC_API_KEY=...  # to switch LLM_PROVIDER to "claude"
```

Leave `laya-serve` and Ollama running, then execute the cells below.

## Config — pick providers here

Switching providers later (e.g. once Jev or a Claude/OpenAI key is available) is a one-line change in this cell — nothing else in the notebook needs to change.

In [1]:
import json
import os
import time

import requests

# --- Decision model provider: "laya" (open-source, local, verified working) | "torchcast" (CUDA-only, NOT usable on Apple Silicon) | "jev" (TypeSafe API) ---
DECISION_PROVIDER = "laya"
LAYA_URL = "http://127.0.0.1:8012"
TORCHCAST_SERVER_URL = "http://127.0.0.1:8011"
JEV_MODEL = "jev-latest"

# --- LLM contrast provider: "ollama" (open-source, local) | "openai" | "claude" ---
LLM_PROVIDER = "ollama"
OLLAMA_URL = "http://localhost:11434"
OLLAMA_MODEL = "qwen3.6"
OPENAI_MODEL = "gpt-4o-mini"
CLAUDE_MODEL = "claude-sonnet-5"

/Users/kfabo/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# Health checks -- fall back to a clearly-labeled SIMULATED response per provider
# if it isn't reachable, so the rest of the notebook still runs end to end.
# The live session should run with both flags False.

LAYA_SIMULATED = True
if DECISION_PROVIDER == "laya":
    try:
        requests.get(f"{LAYA_URL}/health", timeout=2).raise_for_status()
        LAYA_SIMULATED = False
    except requests.RequestException:
        pass

TORCHCAST_SIMULATED = True
if DECISION_PROVIDER == "torchcast":
    try:
        requests.get(f"{TORCHCAST_SERVER_URL}/health", timeout=2).raise_for_status()
        TORCHCAST_SIMULATED = False
    except requests.RequestException:
        pass

OLLAMA_SIMULATED = True
if LLM_PROVIDER == "ollama":
    try:
        requests.get(f"{OLLAMA_URL}/api/version", timeout=2).raise_for_status()
        OLLAMA_SIMULATED = False
    except requests.RequestException:
        pass

_decision_simulated = (DECISION_PROVIDER == "laya" and LAYA_SIMULATED) or (DECISION_PROVIDER == "torchcast" and TORCHCAST_SIMULATED)
print(f"Decision provider: {DECISION_PROVIDER}" + (" [SIMULATED]" if _decision_simulated else " [LIVE]"))
print(f"LLM provider:       {LLM_PROVIDER}" + (" [SIMULATED]" if LLM_PROVIDER == "ollama" and OLLAMA_SIMULATED else " [LIVE]"))

Decision provider: laya [LIVE]
LLM provider:       ollama [LIVE]


## Decision model providers

In [3]:
def decide_laya(state: str, questions: dict) -> tuple[dict, float]:
    """POST to the local laya-serve TypeSafe-compatible /v1/systemone endpoint.
    Verified response shape: {"answers": {<key>: {"noul": <float>, ...}}} for noul questions,
    {"answers": {<key>: {"probabilities": {...}}}} for choice questions.
    """
    start = time.perf_counter()
    if LAYA_SIMULATED:
        time.sleep(0.04)  # representative of laya's self-hosted 30-60ms range
        result = {"_simulated": True, "answers": {k: {"noul": 0.93} for k in questions}}
    else:
        response = requests.post(
            f"{LAYA_URL}/v1/systemone",
            json={"state": {"body": state}, "questions": questions},
            timeout=15,
        )
        response.raise_for_status()
        result = response.json()
    return result, time.perf_counter() - start


def decide_torchcast(state: str, questions: dict) -> tuple[dict, float]:
    """POST to a Torchcast Decision server's TypeSafe-compatible /v1/systemone endpoint.
    NOT usable on Apple Silicon -- see the intro cell. Response shape below is from the
    vendor's documented curl example, unverified against a live server (we never got one
    running): {"<key>": {"noul": {"yes": .., "no": ..}}} for noul, {"<key>": {"choice": {...}}}
    for choice. Adjust if this turns out to differ once run against a real CUDA server.
    """
    start = time.perf_counter()
    if TORCHCAST_SIMULATED:
        time.sleep(0.18)
        result = {"_simulated": True, "decision": {"noul": {"yes": 0.93, "no": 0.07}}}
    else:
        torchcast_questions = {
            k: {**v, "type": "yes_no" if v["type"] == "noul" else v["type"]} for k, v in questions.items()
        }
        response = requests.post(
            f"{TORCHCAST_SERVER_URL}/v1/systemone",
            json={"state": state, "questions": torchcast_questions},
            timeout=10,
        )
        response.raise_for_status()
        result = response.json()
    return result, time.perf_counter() - start


def decide_jev(state: str, questions: dict) -> tuple[dict, float]:
    """POST to TypeSafe's hosted Jev endpoint. Requires TYPESAFE_API_KEY.
    Not used by default -- this is the drop-in path for once Jev access is set up.
    """
    api_key = os.environ.get("TYPESAFE_API_KEY")
    if not api_key:
        raise RuntimeError("Set TYPESAFE_API_KEY to use the Jev provider.")
    start = time.perf_counter()
    response = requests.post(
        "https://api.typesafe.ai/v1/systemone",
        headers={"Authorization": f"Bearer {api_key}"},
        json={"model": JEV_MODEL, "state": state, "questions": questions},
        timeout=10,
    )
    response.raise_for_status()
    return response.json(), time.perf_counter() - start


def decide(state: str, questions: dict) -> tuple[dict, float]:
    if DECISION_PROVIDER == "laya":
        return decide_laya(state, questions)
    if DECISION_PROVIDER == "torchcast":
        return decide_torchcast(state, questions)
    if DECISION_PROVIDER == "jev":
        return decide_jev(state, questions)
    raise ValueError(f"Unknown DECISION_PROVIDER: {DECISION_PROVIDER}")


def extract_noul_prob(result: dict, key: str) -> float:
    """Returns P(yes) for a 'noul' (yes/no) answer, regardless of provider response shape."""
    if DECISION_PROVIDER == "laya":
        return result["answers"][key]["noul"]
    if DECISION_PROVIDER in ("torchcast", "jev"):
        noul = result[key]["noul"]
        return noul["yes"] if isinstance(noul, dict) else noul
    raise ValueError(DECISION_PROVIDER)

## Scenario 1 — Entity resolution

State = two candidate customer/counterparty records. Question = "same entity: yes/no."

In [4]:
record_a = {"name": "Jon A. Smith", "address": "221B Baker St, London", "dob": "1985-03-12", "id_last4": "4471"}
record_b = {"name": "Jonathan Smith", "address": "221 B Baker Street, London", "dob": "1985-03-12", "id_last4": "4471"}

entity_state = (
    "Candidate record A: " + json.dumps(record_a) + "\n"
    "Candidate record B: " + json.dumps(record_b)
)

entity_questions = {
    "decision": {
        "type": "noul",
        "instructions": "Do candidate record A and candidate record B refer to the same real-world entity?",
    }
}

entity_result, entity_elapsed = decide(entity_state, entity_questions)
print(f"elapsed: {entity_elapsed*1000:.0f} ms")
print(f"P(same entity) = {extract_noul_prob(entity_result, 'decision'):.4f}")
print(json.dumps(entity_result, indent=2))

elapsed: 335 ms
P(same entity) = 0.9925
{
  "model": "laya-rl-agent",
  "answers": {
    "decision": {
      "type": "noul",
      "noul": 0.9925,
      "confidence": 0.9925,
      "answer_confidence": 0.9925,
      "action": {
        "act_probability": 1.0
      }
    }
  },
  "usage": {
    "input_tokens": 157,
    "output_tokens": 0,
    "state_tokens": 114,
    "state_tokens_dropped": 0,
    "truncated": false,
    "truncated_questions": []
  },
  "routing": {
    "model": "multilingual",
    "repo": "convaiinnovations/laya/multilingual",
    "reason": "Latin script, language not identified and no non-English letters; using default (multilingual)",
    "detection": {
      "script": "latin",
      "script_profile": {
        "latin": 1.0
      },
      "language": null,
      "is_english": true,
      "language_undecided": true,
      "diacritic_rate": 0.0,
      "non_latin_fraction": 0.0,
      "mixed_segment": null
    },
    "workflow": null
  }
}


## Scenario 2 — Decision-model-as-judge

Same model, same provider, a different question schema: judging a RAG answer against a pass/fail rubric instead of matching entities.

In [5]:
judge_state = json.dumps({
    "question": "What is our policy on reporting transactions above the regulatory threshold?",
    "retrieved_context": (
        "Per internal policy RM-114, transactions exceeding the regulatory threshold "
        "must be reported to the compliance desk within 24 hours of detection."
    ),
    "generated_answer": (
        "Transactions above the regulatory threshold must be reported to the "
        "compliance desk within 24 hours, per policy RM-114."
    ),
})

judge_questions = {
    "decision": {
        "type": "noul",
        "instructions": (
            "Does the generated_answer accurately and completely answer the question, "
            "using only information supported by retrieved_context? Answer pass (yes) or fail (no)."
        ),
    }
}

judge_result, judge_elapsed = decide(judge_state, judge_questions)
print(f"elapsed: {judge_elapsed*1000:.0f} ms")
print(f"P(pass) = {extract_noul_prob(judge_result, 'decision'):.4f}")
print(json.dumps(judge_result, indent=2))

elapsed: 455 ms
P(pass) = 0.8860
{
  "model": "laya-rl-agent",
  "answers": {
    "decision": {
      "type": "noul",
      "noul": 0.886,
      "confidence": 0.886,
      "answer_confidence": 0.886,
      "action": {
        "act_probability": 1.0
      }
    }
  },
  "usage": {
    "input_tokens": 143,
    "output_tokens": 0,
    "state_tokens": 86,
    "state_tokens_dropped": 0,
    "truncated": false,
    "truncated_questions": []
  },
  "routing": {
    "model": "english",
    "repo": "convaiinnovations/laya",
    "reason": "English Latin text",
    "detection": {
      "script": "latin",
      "script_profile": {
        "latin": 1.0
      },
      "language": "en",
      "is_english": true,
      "language_undecided": false,
      "diacritic_rate": 0.0,
      "non_latin_fraction": 0.0,
      "mixed_segment": null
    },
    "workflow": null
  }
}


## LLM contrast providers

Same entity-resolution question, answered by prompting a generic LLM for JSON instead of calling a decision model.

In [6]:
def call_llm_ollama(prompt: str) -> tuple[dict, float]:
    start = time.perf_counter()
    if OLLAMA_SIMULATED:
        time.sleep(1.4)  # representative of a multi-second generate-then-parse round trip
        result = {"_simulated": True, "same_entity": True, "confidence": 0.9}
    else:
        response = requests.post(
            f"{OLLAMA_URL}/api/generate",
            json={
                "model": OLLAMA_MODEL,
                "prompt": prompt,
                "format": "json",
                "stream": False,
                "think": False,  # qwen3.6 is a thinking model -- without this, JSON lands in "thinking" and "response" is empty
            },
            timeout=60,
        )
        response.raise_for_status()
        result = json.loads(response.json()["response"])
    return result, time.perf_counter() - start


def call_llm_openai(prompt: str) -> tuple[dict, float]:
    """Not used by default -- drop-in path once an OPENAI_API_KEY is available."""
    from openai import OpenAI

    client = OpenAI()  # reads OPENAI_API_KEY from env
    start = time.perf_counter()
    response = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
    )
    result = json.loads(response.choices[0].message.content)
    return result, time.perf_counter() - start


def call_llm_claude(prompt: str) -> tuple[dict, float]:
    """Not used by default -- drop-in path once an ANTHROPIC_API_KEY is available."""
    import anthropic

    client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from env
    start = time.perf_counter()
    response = client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=256,
        messages=[{"role": "user", "content": prompt}],
    )
    result = json.loads(response.content[0].text)
    return result, time.perf_counter() - start


def call_llm_for_json(prompt: str) -> tuple[dict, float]:
    if LLM_PROVIDER == "ollama":
        return call_llm_ollama(prompt)
    if LLM_PROVIDER == "openai":
        return call_llm_openai(prompt)
    if LLM_PROVIDER == "claude":
        return call_llm_claude(prompt)
    raise ValueError(f"Unknown LLM_PROVIDER: {LLM_PROVIDER}")

## Contrast — same question, decision model vs. generic LLM

In [7]:
llm_prompt = (
    "Given these two records, answer with ONLY a JSON object "
    '{"same_entity": true|false, "confidence": <0-1 float>}.\n\n'
    f"Record A: {json.dumps(record_a)}\nRecord B: {json.dumps(record_b)}"
)

llm_result, llm_elapsed = call_llm_for_json(llm_prompt)
print(f"elapsed: {llm_elapsed*1000:.0f} ms")
print(json.dumps(llm_result, indent=2))

print("\n--- Timing comparison ---")
print(f"Decision model ({DECISION_PROVIDER}, entity resolution): {entity_elapsed*1000:.0f} ms")
print(f"Generic LLM ({LLM_PROVIDER}) prompted for JSON:          {llm_elapsed*1000:.0f} ms")

elapsed: 34098 ms
{
  "same_entity": true,
  "confidence": 0.98
}

--- Timing comparison ---
Decision model (laya, entity resolution): 335 ms
Generic LLM (ollama) prompted for JSON:          34098 ms


## Discussion prompt

Where in our stack could a decision model replace an LLM-as-judge call or a hand-rolled classifier?